### Dataset and Task Metadata

In [3]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="coil_2000_insurance_policies",
    dataset_year="2000",
    domain_str="business & marketing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5630S",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/coil_2000_insurance_policies/ && wget -P local-data-warehouse/coil_2000_insurance_policies/ https://archive.ics.uci.edu/static/public/125/insurance+company+benchmark+coil+2000.zip && unzip local-data-warehouse/coil_2000_insurance_policies/insurance+company+benchmark+coil+2000.zip -d local-data-warehouse/coil_2000_insurance_policies/
""",
    # References
    academic_reference_bibtex="""@techreport{van2000coil,
  title={CoIL challenge 2000: The insurance company case},
  author={Van Der Putten, Peter and van Someren, Maarten and others},
  year={2000},
  institution={Technical Report 2000--09, Leiden Institute of Advanced Computer Science}
}
""",
    academic_reference_bibtex_key="van2000coil",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We created semantic meaningful names for the features.
- We combined the original training and validation data into one new dataset.
- We reversed the ordinal encoding of the original data where possible.
- Anomaly: the data has 15% duplicates.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="MobileHomePolicy",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="MobileHomePolicy",
)

## Preprocessing

In [4]:
import pandas as pd

data = pd.read_csv(f"{dataset_mold.path}/ticdata2000.txt", sep="\t", header=None)
val_data = pd.concat(
    [
        pd.read_csv(f"{dataset_mold.path}/ticeval2000.txt", sep="\t", header=None),
        pd.read_csv(f"{dataset_mold.path}/tictgts2000.txt", sep="\t", header=None).rename(
            columns={0: 85}
        ),
    ],
    axis=1,
)
df = pd.concat([data, val_data], axis=0, ignore_index=True)

target_feature = "MobileHomePolicy"
df.columns = [
    "customerSubtype",
    "numberOfHouses",
    "avgSizeHousehold",
    "avgAge",
    "customerMainType",
    "romanCatholic",
    "protestant",
    "otherReligion",
    "noReligion",
    "married",
    "livingTogether",
    "otherRelation",
    "singles",
    "householdWithoutChildren",
    "householdWithChildren",
    "highLevelEducation",
    "mediumLevelEducation",
    "lowerLevelEducation",
    "highStatus",
    "entrepreneur",
    "farmer",
    "middleManagement",
    "skilledLabourers",
    "unskilledLabourers",
    "socialClassA",
    "socialClassB1",
    "socialClassB2",
    "socialClassC",
    "socialClassD",
    "rentedHouse",
    "homeOwners",
    "oneCar",
    "twoCars",
    "noCar",
    "nationalHealthService",
    "privateHealthInsurance",
    "incomeLessThan30k",
    "income30To45k",
    "income45To75k",
    "income75To122k",
    "incomeAbove123k",
    "averageIncome",
    "purchasingPowerClass",
    "contributionPrivateThirdPartyInsurance",
    "contributionThirdPartyInsuranceFirms",
    "contributionThirdPartyInsuranceAgriculture",
    "contributionCarPolicies",
    "contributionDeliveryVanPolicies",
    "contributionMotorcycleScooterPolicies",
    "contributionLorryPolicies",
    "contributionTrailerPolicies",
    "contributionTractorPolicies",
    "contributionAgriculturalMachinesPolicies",
    "contributionMopedPolicies",
    "contributionLifeInsurances",
    "contributionPrivateAccidentInsurancePolicies",
    "contributionFamilyAccidentsInsurancePolicies",
    "contributionDisabilityInsurancePolicies",
    "contributionFirePolicies",
    "contributionSurfboardPolicies",
    "contributionBoatPolicies",
    "contributionBicyclePolicies",
    "contributionPropertyInsurancePolicies",
    "contributionSocialSecurityInsurancePolicies",
    "numberOfPrivateThirdPartyInsurance",
    "numberOfThirdPartyInsuranceFirms",
    "numberOfThirdPartyInsuranceAgriculture",
    "numberOfCarPolicies",
    "numberOfDeliveryVanPolicies",
    "numberOfMotorcycleScooterPolicies",
    "numberOfLorryPolicies",
    "numberOfTrailerPolicies",
    "numberOfTractorPolicies",
    "numberOfAgriculturalMachinesPolicies",
    "numberOfMopedPolicies",
    "numberOfLifeInsurances",
    "numberOfPrivateAccidentInsurancePolicies",
    "numberOfFamilyAccidentsInsurancePolicies",
    "numberOfDisabilityInsurancePolicies",
    "numberOfFirePolicies",
    "numberOfSurfboardPolicies",
    "numberOfBoatPolicies",
    "numberOfBicyclePolicies",
    "numberOfPropertyInsurancePolicies",
    "numberOfSocialSecurityInsurancePolicies",
    target_feature,
]

# Reverse ordinal encoding where possible
df["customerSubtype"] = df["customerSubtype"].map({
    1: "High Income, expensive child", 2: "Very Important Provincials",
    3: "High status seniors", 4: "Affluent senior apartments",
    5: "Mixed seniors", 6: "Career and childcare",
    7: "Dinki's (double income no kids)", 8: "Middle class families",
    9: "Modern, complete families", 10: "Stable family",
    11: "Family starters", 12: "Affluent young families",
    13: "Young all american family", 14: "Junior cosmopolitan",
    15: "Senior cosmopolitans", 16: "Students in apartments",
    17: "Fresh masters in the city", 18: "Single youth",
    19: "Suburban youth", 20: "Etnically diverse",
    21: "Young urban have-nots", 22: "Mixed apartment dwellers",
    23: "Young and rising", 24: "Young, low educated",
    25: "Young seniors in the city", 26: "Own home elderly",
    27: "Seniors in apartments", 28: "Residential elderly",
    29: "Porchless seniors: no front yard", 30: "Religious elderly singles",
    31: "Low income catholics", 32: "Mixed seniors",
    33: "Lower class large families", 34: "Large family, employed child",
    35: "Village families", 36: "Couples with teens 'Married with children'",
    37: "Mixed small town dwellers", 38: "Traditional families",
    39: "Large religous families", 40: "Large family farms",
    41: "Mixed rurals",
})
df["avgAge"] = df["avgAge"].map({
    1: "20-30 years", 2: "30-40 years", 3: "40-50 years",
    4: "50-60 years", 5: "60-70 years", 6: "70-80 years",
})
df["customerMainType"] = df["customerMainType"].map({
    1: "Successful hedonists", 2: "Driven Growers", 3: "Average Family",
    4: "Career Loners", 5: "Living well", 6: "Cruising Seniors",
    7: "Retired and Religeous", 8: "Family with grown ups",
    9: "Conservative families", 10: "Farmers",
})
df["romanCatholic"] = df["romanCatholic"].map({
    0: "0%", 1: "1 - 10%", 2: "11 - 23%", 3: "24 - 36%",
    4: "37 - 49%", 5: "50 - 62%", 6: "63 - 75%",
    7: "76 - 88%", 8: "89 - 99%", 9: "100%",
})
df["contributionPrivateThirdPartyInsurance"] = df[
    "contributionPrivateThirdPartyInsurance"
].map({
    0: "f 0", 1: "f 1 - 49", 2: "f 50 - 99", 3: "f 100 - 199",
    4: "f 200 - 499", 5: "f 500 - 999", 6: "f 1000 - 4999",
    7: "f 5000 - 9999", 8: "f 10.000 - 19.999", 9: "f 20.000 - ?",
})
df[target_feature] = df[target_feature].map({0: "No", 1: "Yes"})

cat_features = [
    "customerSubtype",
    "customerMainType",
    "romanCatholic",
    "MobileHomePolicy",
    "avgAge",
    "contributionPrivateThirdPartyInsurance",
]
df[cat_features] = df[cat_features].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [5]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 9,822
Columns: 86
Use sampling: False (sample size: 9,822)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['customerSubtype', 'homeOwners', 'skilledLabourers', 'unskilledLabourers', 'socialClassA', 'socialClassB1', 'socialClassB2', 'socialClassC', 'socialClassD', 'rentedHouse']
Rows remaining as candidates after top-10 filter: 8,894 (of 9,822)

#### Duplicate Report
Total duplicate rows: 1442 (14.68% of dataset)
Duplicate rows ignoring target: 1561 (15.89% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [6]:
# Sample Rows
df_head

,customerSubtype,numberOfHouses,avgSizeHousehold,avgAge,customerMainType,romanCatholic,protestant,otherReligion,noReligion,married,livingTogether,otherRelation,singles,householdWithoutChildren,householdWithChildren,highLevelEducation,mediumLevelEducation,lowerLevelEducation,highStatus,entrepreneur,farmer,middleManagement,skilledLabourers,unskilledLabourers,socialClassA,socialClassB1,socialClassB2,socialClassC,socialClassD,rentedHouse,homeOwners,oneCar,twoCars,noCar,nationalHealthService,privateHealthInsurance,incomeLessThan30k,income30To45k,income45To75k,income75To122k,incomeAbove123k,averageIncome,purchasingPowerClass,contributionPrivateThirdPartyInsurance,contributionThirdPartyInsuranceFirms,contributionThirdPartyInsuranceAgriculture,contributionCarPolicies,contributionDeliveryVanPolicies,contributionMotorcycleScooterPolicies,contributionLorryPolicies,contributionTrailerPolicies,contributionTractorPolicies,contributionAgriculturalMachinesPolicies,contributionMopedPolicies,contributionLifeInsurances,contributionPrivateAccidentInsurancePolicies,contributionFamilyAccidentsInsurancePolicies,contributionDisabilityInsurancePolicies,contributionFirePolicies,contributionSurfboardPolicies,contributionBoatPolicies,contributionBicyclePolicies,contributionPropertyInsurancePolicies,contributionSocialSecurityInsurancePolicies,numberOfPrivateThirdPartyInsurance,numberOfThirdPartyInsuranceFirms,numberOfThirdPartyInsuranceAgriculture,numberOfCarPolicies,numberOfDeliveryVanPolicies,numberOfMotorcycleScooterPolicies,numberOfLorryPolicies,numberOfTrailerPolicies,numberOfTractorPolicies,numberOfAgriculturalMachinesPolicies,numberOfMopedPolicies,numberOfLifeInsurances,numberOfPrivateAccidentInsurancePolicies,numberOfFamilyAccidentsInsurancePolicies,numberOfDisabilityInsurancePolicies,numberOfFirePolicies,numberOfSurfboardPolicies,numberOfBoatPolicies,numberOfBicyclePolicies,numberOfPropertyInsurancePolicies,numberOfSocialSecurityInsurancePolicies,MobileHomePolicy
0,Low income catholics,1,3,40-50 years,Retired and Religeous,0%,2,4,4,9,0,0,0,3,6,0,4,5,1,0,0,3,3,2,2,2,2,5,1,9,0,5,3,2,6,3,0,4,5,1,0,4,1,f 0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,No
1,Stable family,1,4,40-50 years,Average Family,50 - 62%,3,1,1,8,1,1,1,1,8,1,5,3,1,0,0,5,3,1,1,3,3,3,0,1,8,9,0,0,5,4,3,2,4,0,0,4,8,f 50 - 99,0,0,6,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,No
2,Large family farms,1,3,30-40 years,Farmers,0%,6,0,3,6,3,0,0,5,4,1,7,2,3,0,3,3,0,1,4,2,2,2,0,0,9,4,5,0,3,6,0,4,4,2,0,5,3,f 0,0,0,7,0,0,0,2,0,0,0,0,0,0,0,6,0,0,0,0,0,0,0,0,3,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,No
3,Mixed seniors,2,2,60-70 years,Successful hedonists,0%,4,2,3,5,0,4,4,4,2,4,0,5,4,0,0,0,4,2,4,0,0,5,0,4,5,5,0,4,5,4,1,3,5,1,0,5,3,f 0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,No
4,Lower class large families,2,3,30-40 years,Family with grown ups,0%,7,0,2,7,2,0,0,4,5,0,2,7,0,2,0,2,4,2,0,0,4,5,0,2,7,5,4,0,9,0,0,5,4,0,0,4,3,f 0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,No


In [7]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,customerSubtype,category,0.0,0.0,39.0,"Lower class large families, Traditional families, Middle class families, Large religous families, Modern, complete families, High status seniors, Young and rising, Couples with teens 'Married with children', Village families, Mixed rurals"
1,avgAge,category,0.0,0.0,6.0,"40-50 years, 30-40 years, 50-60 years, 60-70 years, 20-30 years, 70-80 years"
2,customerMainType,category,0.0,0.0,10.0,"Family with grown ups, Average Family, Conservative families, Successful hedonists, Living well, Retired and Religeous, Driven Growers, Farmers, Cruising Seniors, Career Loners"
3,romanCatholic,category,0.0,0.0,10.0,"0%, 1 - 10%, 11 - 23%, 24 - 36%, 37 - 49%, 50 - 62%, 63 - 75%, 76 - 88%, 100%, 89 - 99%"
4,contributionPrivateThirdPartyInsurance,category,0.0,0.0,4.0,"f 0, f 50 - 99, f 1 - 49, f 100 - 199"
5,MobileHomePolicy,category,0.0,0.0,2.0,"No, Yes"
6,numberOfHouses,int64,0.0,0.0,9.0,"1, 2, 3, 7, 4, 6, 5, 10, 8"
7,avgSizeHousehold,int64,0.0,0.0,6.0,"3, 2, 4, 1, 5, 6"
8,protestant,int64,0.0,0.0,10.0,"4, 5, 6, 3, 7, 2, 9, 1, 0, 8"
9,otherReligion,int64,0.0,0.0,6.0,"0, 1, 2, 3, 4, 5"


In [8]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
numberOfHouses,9822.0,1.108735,0.412101,1.0,10.0
avgSizeHousehold,9822.0,2.677561,0.780701,1.0,6.0
protestant,9822.0,4.637650,1.721212,0.0,9.0
otherReligion,9822.0,1.050092,1.011156,0.0,5.0
noReligion,9822.0,3.262981,1.606287,0.0,9.0
married,9822.0,6.188964,1.896070,0.0,9.0
livingTogether,9822.0,0.873142,0.961955,0.0,7.0
otherRelation,9822.0,2.286602,1.710674,0.0,9.0
singles,9822.0,1.887294,1.779238,0.0,9.0
householdWithoutChildren,9822.0,3.237324,1.609139,0.0,9.0


In [9]:
# Categorical Feature Statistics
cat_stats

value  \
column                                 rank                               
MobileHomePolicy                       1                             No   
                                       2                            Yes   
avgAge                                 1                    40-50 years   
                                       2                    30-40 years   
                                       3                    50-60 years   
                                       4                    60-70 years   
                                       5                    20-30 years   
contributionPrivateThirdPartyInsurance 1                            f 0   
                                       2                      f 50 - 99   
                                       3                       f 1 - 49   
                                       4                    f 100 - 199   
customerMainType                       1          Family with grown ups   
                                       2                 Average Family   
                                       3          Conservative families   
                                       4           Successful hedonists   
                                       5                    Living well   
customerSubtype                        1     Lower class large families   
                                       2           Traditional families   
                                       3          Middle class families   
                                       4        Large religous families   
                                       5      Modern, complete families   
romanCatholic                          1                             0%   
                                       2                        1 - 10%   
                                       3                       11 - 23%   
                                       4                       24 - 36%   
                                       5                       37 - 49%   

                                             count    pct  
column                                 rank                
MobileHomePolicy                       1      9236  94.03  
                                       2       586   5.97  
avgAge                                 1      5154  52.47  
                                       2      2409  24.53  
                                       3      1777  18.09  
                                       4       329   3.35  
                                       5       104   1.06  
contributionPrivateThirdPartyInsurance 1      5903  60.10  
                                       2      3562  36.27  
                                       3       341   3.47  
                                       4        16   0.16  
customerMainType                       1      2694  27.43  
                                       2      1513  15.40  
                                       3      1111  11.31  
                                       4       959   9.76  
                                       5       940   9.57  
customerSubtype                        1      1401  14.26  
                                       2       569   5.79  
                                       3       546   5.56  
                                       4       542   5.52  
                                       5       460   4.68  
romanCatholic                          1      5420  55.18  
                                       2      2744  27.94  
                                       3      1213  12.35  
                                       4       243   2.47  
                                       5       123   1.25

In [10]:
# Target Distribution
target_df

,count,pct
MobileHomePolicy,,
No,9236,94.03
Yes,586,5.97


## Task Curation

In [11]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [12]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [13]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to coil_2000_insurance_policies/019d9cd5-876e-7cd5-876c-2edf17c699e9
019d9cd5-876e-7cd5-876c-2edf17c699e9
fa35f1ebaacb959a784d354206d9215e55328511f1c338317a39bb8a9f84005d
